# Spatial Prediction of Energy Poverty Indicators

Train a separate ridge, random forest and histogram gradient boosting model for each
target. Predictors remain satellite indicators plus the **basic** administrative
indicators in `data/adm_data_split.json`. Detailed administrative indicators are
reserved for residual analysis; private inputs and fitted outputs stay outside git.

This notebook orchestrates `pipeline.modeling`. Model recipes, training, evaluation
and artifact storage are independently defined there. Saved spatial assignments come
from `3_0_spatial_test_set_selection.ipynb`; training never rebuilds them.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "pipeline" / "config").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pipeline.modeling import (
    DEFAULT_TARGETS, TargetSpec, EvaluationSpec, SearchConfig,
    load_project_data, preview_experiment, run_experiment,
)
from pipeline.utils.paths import load_paths, path_value

pd.set_option("display.max_columns", 100)
PATHS = load_paths()

## Experiment Configuration

Add a target with `TargetSpec("stable_key", "Source column", units="optional units")`.
The column must exist in the configured target CSV. Missing outcomes exclude rows
only for that target; every model for that target uses exactly the same rows/folds.
For a target from another source, extend dataset assembly, not the model recipes.

In [ ]:
TARGETS = list(DEFAULT_TARGETS)
MODELS = ["ridge", "random_forest", "histogram_boosting"]
EXCLUDE_OVERSEAS_TERRITORIES = True

EVALUATION = EvaluationSpec(
    metrics=("rmse", "mae", "r2", "spearman"),
    primary="rmse",
)
SEARCH = SearchConfig(seed=42, n_iter=40, n_jobs=2, verbose=1)
OUTPUT_ROOT = path_value(PATHS, "external_data_root") / "outputs/epvi_prediction/runs"

## Load and Validate Data

Names and spatial metadata come from explicit assignment columns. Missing requested
targets, duplicate parish IDs, prohibited predictors, invalid numeric values and
region overlap fail before fitting. Empty/constant predictors are filtered inside
each fitted pipeline, not using the combined training and test data.

In [ ]:
dataset = load_project_data(TARGETS, exclude_overseas=EXCLUDE_OVERSEAS_TERRITORIES)
print("Scope:", dataset.scope)
print("Rows:", len(dataset.frame), "| Candidate predictors:", len(dataset.predictors))
display(pd.DataFrame(dataset.coverage))
display(dataset.frame.groupby(["spatial_split_role", "spatial_cv_fold"], dropna=False)
        .agg(n_rows=("ID_norm", "size"), n_regions=("NUTS3", "nunique")))
display(dataset.frame[["ID_norm", "NUTS3", "spatial_split_role", "spatial_cv_fold",
                       *[target.column for target in TARGETS]]].head())

## Modeling Design

For each model-target pair, search hyperparameters using the saved training-region
spatial folds, refit the selected pipeline on all training rows, and predict the fixed
test set. Ridge uses median imputation and standardization; both tree models use
median imputation without scaling. Boosting disables internal early stopping: spatial
CV selects its iteration count. All learned preprocessing is fitted inside CV.

The primary tuning objective is **unweighted mean spatial-fold RMSE** (lower is better),
an explicit change from historical mean-fold R² tuning. All models report RMSE, MAE,
R² and Spearman on the original target scale. Raw errors are not averaged across
different targets. Fold means and metrics on pooled OOF predictions are distinct.

**OOF limitation (intentional):** Spatial OOF predictions exclude each observation's
fold from model fitting, but use hyperparameters previously selected using all training
folds. They are therefore not independent of hyperparameter selection and may yield
optimistic performance estimates. These predictions and residuals are used for
diagnostics; fixed-test results provide the held-out assessment. Nested CV is not run.

A training-mean baseline is fitted separately within each OOF training partition and
on the full training set for test prediction. The final fitted pipeline is already
produced by the search refit; it is not fitted again for test evaluation.

The holdout has been inspected in earlier research and was previously replaced;
see `docs/model_tuning_history.md`. Keep its assignment frozen and disclose that history
rather than describing it as a never-inspected confirmatory test.

In [ ]:
plan = preview_experiment(dataset, models=MODELS, targets=TARGETS,
                          evaluation=EVALUATION, search=SEARCH)
display(plan.drop(columns="search_space"))
for row in plan.itertuples():
    print(f"{row.target} / {row.model}: {row.search_space}")
print("Estimated model fits:", plan.total_fits_with_oof_and_refit.sum())
print("Plus one training-mean baseline per target (one fit per fold and one full fit).")

## Run Experiment

Each completed model-target pair is saved immediately, with its fitted pipeline,
search results, predictions, residuals, overall/fold/regional metrics and optional
model-specific diagnostics. The run manifest records the full configuration,
input/split hashes, code provenance and package versions. An interrupted run retains
completed pairs and can be inspected with `load_run(path, allow_incomplete=True)`.

In [ ]:
results = run_experiment(
    dataset, models=MODELS, targets=TARGETS,
    evaluation=EVALUATION, search=SEARCH, output_root=OUTPUT_ROOT,
)
print("Run directory for the performance notebook:", results.run_dir)

## Compare Results

Compare models within each target. Test results are held-out assessment; use training
spatial CV for tuning decisions. Undefined metrics remain missing with a reason in the
long-form metrics table (for example, Spearman for constant baseline predictions).

In [ ]:
display(results.comparison_table("fixed_test"))
display(results.comparison_table("train_spatial_cv"))
display(results.metrics.loc[results.metrics.undefined_reason.notna()])

## Next Step

Open `3_2_model_performance.ipynb` with the exact run ID printed above. Its model
comparison uses one run manifest, avoiding accidental mixing of separately selected
latest files. Ridge coefficients and forest impurity importances are separate
diagnostics, not a common importance scale. The historical RF output directory and
legacy EPG diagnostics remain available for earlier runs.